In [5]:
import numpy as np
import pandas as pd
from pathlib import Path

# -------- paths --------
SIM = Path("../../../data/simulation/")

INPUTS = [
    ("engine_normal_load_X.npy",   "engine_normal_load_y.npy"),
    ("engine_high_load_X.npy",     "engine_high_load_y.npy"),
    ("engine_critical_load_X.npy", "engine_critical_load_y.npy"),
    ("engine_off_X.npy",           "engine_off_y.npy"),
    ("engine_start_X.npy",         "engine_start_y.npy"),
]

OUT_X   = SIM / "engine_occ_X.npy"
OUT_Y   = SIM / "engine_occ_y.npy"
OUT_CSV = SIM / "engine_occ.csv"

FEATURES = ["Temperature","Pressure","RPM","Vibration"]
rng = np.random.default_rng(42)

# -------- Uncalibrated value zones --------
# The rule-based input check only catches EXTREME values; values in the
# MODERATE zone pass the rules and must be caught by M0. Labels are the
# same everywhere: "Uncalibrated" is one concept with two catchers.
#   Temperature  moderate 146.0..164.9 / -29.9..-21.0 | extreme 165..300 / -90..-30
#   Pressure     moderate 1.30..1.45  /  0.05..0.45   | extreme 1.5..6  / -6..0.01
#   RPM          moderate 9500..11900                  | extreme 12000..100000 / negative
#   Vibration    moderate 0.80..1.90                   | extreme 2..20 / negative
MODERATE_PROB = 0.30  # 30% moderate / 70% extreme in the simulation

def _oor_moderate(name, size):
    if name == "Temperature":
        return (rng.uniform(-29.9, -21.0, size) if rng.random() < 0.5 else rng.uniform(146.0, 164.9, size))
    if name == "Pressure":
        return (rng.uniform(0.05, 0.45, size) if rng.random() < 0.5 else rng.uniform(1.30, 1.45, size))
    if name == "RPM":
        return rng.uniform(9500.0, 11900.0, size)
    return rng.uniform(0.80, 1.90, size)

def _oor_extreme(name, size):
    if name == "Temperature":
        return (rng.uniform(-90, -30, size) if rng.random() < 0.5 else rng.uniform(165, 300, size))
    if name == "Pressure":
        return (rng.uniform(-6, 0.01, size) if rng.random() < 0.5 else rng.uniform(1.5, 6, size))
    if name == "RPM":
        return (rng.uniform(-2000, -0.1, size) if rng.random() < 0.5 else rng.uniform(12000, 100000, size))
    return (rng.uniform(-20, -0.1, size) if rng.random() < 0.5 else rng.uniform(2, 20, size))

def _oor(name, size):
    if rng.random() < MODERATE_PROB:
        return _oor_moderate(name, size)
    return _oor_extreme(name, size)

def err_out_of_range(col, feat):
    # Entire column becomes out-of-range (sensor drift / bad calibration)
    return _oor(feat, col.shape[0]).astype(col.dtype)

def err_stuck(col, feat):
    # Entire column becomes a flat line (stuck sensor)
    return np.full_like(col, float(col[1] if col.shape[0] > 1 else col[0]))

def err_error_value(col, feat):
    """
    Same logic as before (scatter vs chunk), but instead of numeric sentinel (-999999.0),
    we insert a textual placeholder (gibberish/labels).
    All non-sentinel positions are replaced with out-of-range numeric 'uncalibrated' values.
    """
    GIBRISH_POOL = [
        "Error", "Loading..", "-------", "", "Sensor Error",
        "udh98u8u832y2+28«3'", '=")/)(UI)IW)IW)"IM'
    ]
    PROB_CHUNK = 0.05            # ~5% of sequences use chunk mode
    PROB_SENTINEL_SCATTER = 0.35 # density of sentinels in scatter mode (per-timestep target)
    MAX_CHUNKS = 4               # cap number of chunks in chunk mode

    size = col.shape[0]
    uncalibrated = _oor(feat, size).astype(np.float32)
    out = uncalibrated.astype(object)  # allow mixing floats + strings

    if rng.random() < PROB_CHUNK:
        # ---- Chunk mode: 1–3 chunks, length 2..5, non-overlapping and non-touching ----
        sentinel_mask = np.zeros(size, dtype=bool)
        max_possible_chunks = min(MAX_CHUNKS, max(1, size // 10))
        n_chunks = int(rng.integers(1, max_possible_chunks + 1))
        attempts = 0
        while n_chunks > 0 and attempts < 100:
            attempts += 1
            L = int(rng.integers(2, 6))  # 2..5 inclusive
            if L >= size:
                continue
            start = int(rng.integers(0, size - L + 1))
            end = start + L
            # avoid overlap or touching existing chunks (gap >= 1)
            if sentinel_mask[max(0, start-1):min(size, end+1)].any():
                continue
            sentinel_mask[start:end] = True
            n_chunks -= 1
        # place text sentinels in the chunked positions
        idxs = np.where(sentinel_mask)[0]
        if idxs.size:
            out[idxs] = rng.choice(GIBRISH_POOL, size=idxs.size)
    else:
        # ---- Scatter mode: place isolated sentinels with no adjacency ----
        sentinel_mask = np.zeros(size, dtype=bool)
        target = int(round(PROB_SENTINEL_SCATTER * size))
        candidates = rng.permutation(size)
        placed = 0
        for idx in candidates:
            if placed >= target:
                break
            if (idx > 0 and sentinel_mask[idx-1]) or (idx < size-1 and sentinel_mask[idx+1]):
                continue
            sentinel_mask[idx] = True
            placed += 1
        idxs = np.where(sentinel_mask)[0]
        if idxs.size:
            out[idxs] = rng.choice(GIBRISH_POOL, size=idxs.size)

    return out

ERROR_FUNCS = {"out_of_range": err_out_of_range, "stuck": err_stuck, "error_value": err_error_value}
ERROR_NAMES = ["out_of_range", "stuck", "error_value"]
ERROR_PROBS = np.array([0.65, 0.10, 0.25], dtype=float)

def apply_occ(win_T4):
    """
    Apply OCC to a (T,4) window.
    To support string sentinels for 'error_value', we operate on an object array.
    """
    # work on object dtype so we can mix floats and strings transparently
    w_base = win_T4.astype(np.float32, copy=True)
    w = w_base.astype(object, copy=True)

    k = int(rng.integers(1, 5))
    var_idx = rng.choice(4, size=k, replace=False)
    for vi in var_idx:
        feat = FEATURES[vi]
        names = ERROR_NAMES.copy(); probs = ERROR_PROBS.copy()
        if feat == "RPM" and np.allclose(w_base[:, vi].astype(np.float32), 0.0):
            # Avoid labeling zero-RPM windows as 'stuck' (meaningless for Off)
            j = names.index("stuck")
            names.pop(j)
            probs = np.delete(probs, j)
            probs = probs / probs.sum()
        chosen = rng.choice(names, p=probs)

        # Run the chosen corruption function on the float32 base column
        corrupted = ERROR_FUNCS[chosen](w_base[:, vi], feat)
        # Assign into object array (supports both numeric and string outputs)
        w[:, vi] = corrupted

    return w

# -------- build datasets --------
X_total = []
y_total = []

for x_name, y_name in INPUTS:
    X = np.load(SIM / x_name, allow_pickle=True)
    _ = np.load(SIM / y_name, allow_pickle=True)  # just to confirm file exists

    assert X.ndim == 3 and X.shape[2] == 4, f"Bad X shape {X.shape} @ {x_name}"

    for i in range(X.shape[0]):
        w = X[i]             # (T,4)
        w_occ = apply_occ(w) # OCC filtered (dtype=object)
        T = w_occ.shape[0]

        X_total.append(w_occ)

        # Label handling (all Unknown for OCC)
        if "engine_off" in x_name.lower():
            y_total.append(np.repeat("Unknown", 30))  # force (30,)
        else:
            y_total.append(np.repeat("Unknown", T))   # match sequence length

# -------- save ragged arrays --------
X_total = np.array(X_total, dtype=object)
y_total = np.array(y_total, dtype=object)

np.save(OUT_X, X_total, allow_pickle=True)
np.save(OUT_Y, y_total, allow_pickle=True)

# -------- CSV export --------
def _csv_val(v):
    # Keep strings as-is; cast numeric-like to float.
    # Handles numpy scalars and Python numbers.
    if isinstance(v, (np.floating, float, int, np.integer)):
        return float(v)
    # try to coerce numerics stored as strings (optional)
    try:
        return float(v)
    except Exception:
        return str(v)

rows = []
for i, (X_seq, y_seq) in enumerate(zip(X_total, y_total)):
    for t in range(X_seq.shape[0]):
        rows.append({
            "Sequence":   i + 1,
            "Time":       t + 1,
            "Temperature": _csv_val(X_seq[t, 0]),
            "Pressure":    _csv_val(X_seq[t, 1]),
            "RPM":         _csv_val(X_seq[t, 2]),
            "Vibration":   _csv_val(X_seq[t, 3]),
            "State":       y_seq[t],
        })

pd.DataFrame(
    rows,
    columns=["Sequence","Time","Temperature","Pressure","RPM","Vibration","State"]
).to_csv(OUT_CSV, index=False)

print("X_total:", len(X_total), "| Example shapes:", [x.shape for x in X_total[:5]])

X_total: 3325 | Example shapes: [(30, 4), (30, 4), (30, 4), (30, 4), (30, 4)]
